In [15]:
import pandas as pd

file_path_loads = "Load_profiles_Freight transports.xlsx"
load_profile = pd.read_excel(file_path_loads)
for time in load_profile:
    print(time)
load_profile["Datetime"] = pd.to_datetime(
    load_profile["Date"].astype(str) + " " + load_profile["Time"].astype(str)
).dt.tz_localize(None)
load_profile = load_profile.set_index("Datetime")
shops = ["ICA", "Mathem", "Postnord", "Airmee"]
load_profile = load_profile[shops]

print(load_profile["Airmee"])
load = load_profile["Airmee"] / 1000.0
print(load)
load = load.to_dict()
print(load)

Date
Time
ICA
Mathem
Postnord
Airmee
Datetime
2026-01-01 00:00:00    179.548523
2026-01-01 01:00:00    161.847819
2026-01-01 02:00:00    174.293495
2026-01-01 03:00:00    149.116478
2026-01-01 04:00:00    148.731466
                          ...    
2026-01-14 19:00:00    433.393457
2026-01-14 20:00:00    435.872790
2026-01-14 21:00:00    356.205457
2026-01-14 22:00:00    336.148495
2026-01-14 23:00:00    357.541321
Name: Airmee, Length: 336, dtype: float64
Datetime
2026-01-01 00:00:00    0.179549
2026-01-01 01:00:00    0.161848
2026-01-01 02:00:00    0.174293
2026-01-01 03:00:00    0.149116
2026-01-01 04:00:00    0.148731
                         ...   
2026-01-14 19:00:00    0.433393
2026-01-14 20:00:00    0.435873
2026-01-14 21:00:00    0.356205
2026-01-14 22:00:00    0.336148
2026-01-14 23:00:00    0.357541
Name: Airmee, Length: 336, dtype: float64
{Timestamp('2026-01-01 00:00:00'): 0.1795485235, Timestamp('2026-01-01 01:00:00'): 0.1618478192, Timestamp('2026-01-01 02:00:00'): 0.17

In [ ]:
# ------------- 1. create network and apply initial change -------------
net0 = nw.create_cigre_network_mv(with_der=False)

net0.switch["closed"] = True  # close the tie switch

# scale down loads
net0.load['p_mw'] = net0.load['p_mw'] / 2.0
net0.load['q_mvar'] = net0.load['q_mvar'] / 2.0

base_loads = net0.load['p_mw'].copy()




# Initialize shop placements
shop_to_bus = {
    "ICA": None,
    "Mathem": None,
    "Postnord": None,
    "Airmee": None,
}



# Function to create network with shop loads
def create_network_with_shops(net_base, shop_mapping, timestamp, charging_loads):
    """Create a network with base loads + shop loads at specified buses"""
    net = copy.deepcopy(net_base)
    
    # Reset to base loads first
    net.load['p_mw'] = base_loads.copy()
    
    # Add shop loads to their respective buses
    for shop, bus in shop_mapping.items():
        if bus is not None:
            # Find if there's already a load at this bus
            existing_loads = net.load[net.load.bus == bus]
            
            if not existing_loads.empty:
                # Add to existing load
                load_idx = existing_loads.index[0]
                add_mw_freight = load_profile.at[timestamp, shop] / 1000.0
                net.load.at[load_idx, 'p_mw'] += add_mw_freight
                add_mw_charging = charging_loads.at[timestamp, shop] / 1000.0
                net.load.at[load_idx, 'p_mw'] += add_mw_charging
            else:
                # Create new load at this bus
                add_mw_freight = load_profile.at[timestamp, shop] / 1000.0
                add_mw_charging = charging_loads.at[timestamp, shop] / 1000.0
                add_mw = add_mw_charging + add_mw_freight
                pp.create_load(net, bus=bus, p_mw=add_mw, q_mvar=0)
    
    return net


      


# ------------- 6. Run final simulation with all shops -------------
print("Running final simulation with all shops...")

line_indices = list(net0.line.index)
col_names = [f"line_{i}" for i in line_indices]
res_lines_final_losses = pd.DataFrame(index=load_profile.index, columns=col_names, dtype=float)




all_losses = 0.0

for ts_idx, ts in enumerate(load_profile.index):
    if ts_idx % 24 == 0:  # Print progress every 24 hours
        print(f"Final simulation progress: {ts}")
    
    net_final = create_network_with_shops(net0, shop_to_bus, ts, charging_loads)

    try:
        pp.runpp(net_final, calculate_voltage_angles=False)
        line_loading = net_final.res_line.loading_percent 
        line_losses = net_final.res_line.pl_mw
        trafo_loading = net_final.res_trafo.loading_percent
        bus_voltage = net_final.res_bus.vm_pu
        for li in line_indices:
            res_lines_final_losses.at[ts, f"line_{li}"] = line_losses.at[li]
            all_losses += line_losses.at[li]



    except Exception as e:
        print(f"Final power flow failed at {ts}: {e}")
        


print(all_losses)